# BERT Model Training with Validation

## Objective
Train a BERT model for fake news classification with proper validation tracking and metrics evaluation.

**Key Focus**: Proper use of validation dataset during training, best model selection, and comprehensive evaluation.

---

## Pipeline Overview
1. Load preprocessed data and model components from setup notebooks
2. Define custom Trainer with weighted loss
3. Configure training arguments with validation
4. Implement metrics computation (accuracy, precision, recall, F1)
5. Train model with validation tracking
6. Evaluate on validation set
7. Test on test set (no data leakage)

In [1]:
# Import Required Libraries
import pickle
import json
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# PyTorch and HuggingFace
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
import numpy as np

# Seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Device: {device}")
print(f"  PyTorch version: {torch.__version__}")
print(f"  CUDA available: {torch.cuda.is_available()}")

✓ Device: cpu
  PyTorch version: 2.5.1+cpu
  CUDA available: False


## Section 1: Load Preprocessed Data and Model Components

Load tokenized datasets, label mappings, model, and loss function from previous setup.

In [10]:
print("=" * 80)
print("LOADING PREPROCESSED DATA AND MODEL COMPONENTS")
print("=" * 80)

# Load preprocessed data
preprocessed_dir = Path("preprocessed_data")

print("\nLoading tokenized datasets...")
with open(preprocessed_dir / "train_tokenized.pkl", "rb") as f:
    train_tokenized = pickle.load(f)
with open(preprocessed_dir / "valid_tokenized.pkl", "rb") as f:
    valid_tokenized = pickle.load(f)
with open(preprocessed_dir / "test_tokenized.pkl", "rb") as f:
    test_tokenized = pickle.load(f)

print(f"  ✓ Training: {train_tokenized['input_ids'].shape[0]} samples")
print(f"  ✓ Validation: {valid_tokenized['input_ids'].shape[0]} samples")
print(f"  ✓ Test: {test_tokenized['input_ids'].shape[0]} samples")

# Load label mappings
with open(preprocessed_dir / "label_mappings.json", "r") as f:
    mappings = json.load(f)
    label_to_id = mappings["label_to_id"]
    id_to_label = {int(k): v for k, v in mappings["id_to_label"].items()}
    MAX_LENGTH = mappings["max_length"]

print(f"\n✓ Label mappings loaded")
print(f"  Labels: {list(label_to_id.keys())}")

# Custom Dataset class for dict-based format
from torch.utils.data import Dataset

class TokenizedDataset(Dataset):
    """Custom dataset class that returns dictionaries instead of tuples"""
    
    def __init__(self, tokenized_data):
        self.input_ids = tokenized_data['input_ids']
        self.attention_mask = tokenized_data['attention_mask']
        self.labels = tokenized_data['labels']
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return {
            'input_ids': torch.tensor(self.input_ids[idx], dtype=torch.long),
            'attention_mask': torch.tensor(self.attention_mask[idx], dtype=torch.long),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# Create datasets using custom dataset class
print("\nCreating datasets...")
train_dataset = TokenizedDataset(train_tokenized)
eval_dataset = TokenizedDataset(valid_tokenized)
test_dataset = TokenizedDataset(test_tokenized)

print(f"  ✓ Train dataset: {len(train_dataset)} samples")
print(f"  ✓ Eval dataset: {len(eval_dataset)} samples")
print(f"  ✓ Test dataset: {len(test_dataset)} samples")

# Load tokenizer
print("\nLoading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
print(f"  ✓ Tokenizer loaded")

# Load model
print("\nLoading BERT model...")
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(id_to_label)
)
model.to(device)
print(f"  ✓ Model loaded: {type(model).__name__}")
print(f"  ✓ Total parameters: {sum(p.numel() for p in model.parameters()):,}")

# Compute class weights
print("\nComputing class weights...")
from sklearn.utils.class_weight import compute_class_weight

train_labels = train_tokenized['labels']
class_weights_np = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)
class_weights_tensor = torch.tensor(class_weights_np, dtype=torch.float32, device=device)

print(f"  ✓ Class weights:")
for label_id in range(len(id_to_label)):
    print(f"    {id_to_label[label_id]}: {class_weights_np[label_id]:.4f}")

print(f"\n✓ All components loaded successfully")

LOADING PREPROCESSED DATA AND MODEL COMPONENTS

Loading tokenized datasets...
  ✓ Training: 10240 samples
  ✓ Validation: 1284 samples
  ✓ Test: 1267 samples

✓ Label mappings loaded
  Labels: ['False', 'Nuanced', 'True']

Creating datasets...
  ✓ Train dataset: 10240 samples
  ✓ Eval dataset: 1284 samples
  ✓ Test dataset: 1267 samples

Loading tokenizer...


loading configuration file config.json from cache at C:\Users\Administrateur\.cache\huggingface\hub\models--bert-base-uncased\snapshots\86b5e0934494bd15c9632b12f734a8a67f723594\config.json
Model config BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "eos_token_id": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "tie_word_embeddings": true,
  "transformers_version": "5.4.0",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

loading configuration file config.json from cache at C:\U

  ✓ Tokenizer loaded

Loading BERT model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  ✓ Model loaded: BertForSequenceClassification
  ✓ Total parameters: 109,484,547

Computing class weights...
  ✓ Class weights:
    False: 1.2044
    Nuanced: 0.9059
    True: 0.9382

✓ All components loaded successfully


## Section 2: Define Custom Trainer with Weighted Loss

Create a custom Trainer class that overrides compute_loss to use weighted CrossEntropyLoss.

In [14]:
print("=" * 80)
print("DEFINING CUSTOM TRAINER WITH WEIGHTED LOSS")
print("=" * 80)

class WeightedLossTrainer(Trainer):
    """
    Custom Trainer class that uses weighted CrossEntropyLoss
    to handle class imbalance in the training process.
    """
    
    def __init__(self, class_weights, *args, **kwargs):
        """
        Initialize trainer with class weights.
        
        Args:
            class_weights: torch.Tensor of shape (num_labels,) containing class weights
        """
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        
        # Initialize loss function with class weights
        self.loss_fn = nn.CrossEntropyLoss(weight=self.class_weights)
        
        print(f"\n✓ WeightedLossTrainer initialized")
        print(f"  Class weights: {self.class_weights.cpu().numpy()}")
        print(f"  Loss function: CrossEntropyLoss (weighted)")
    
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        """
        Override compute_loss to use weighted CrossEntropyLoss.
        
        Args:
            model: The model to compute loss for
            inputs: Dictionary containing input_ids, attention_mask, labels
            return_outputs: Whether to return model outputs
            num_items_in_batch: Number of items in batch (for newer transformers versions)
        
        Returns:
            loss or (loss, outputs) tuple
        """
        # Extract labels from inputs
        labels = inputs.pop("labels")
        
        # Forward pass through model
        outputs = model(**inputs)
        logits = outputs.logits
        
        # Compute weighted loss
        loss = self.loss_fn(logits, labels)
        
        # Ensure weights are on the same device as loss
        if loss.device != self.class_weights.device:
            self.class_weights = self.class_weights.to(loss.device)
        
        return (loss, outputs) if return_outputs else loss

print("\n✓ Custom Trainer class defined successfully")

DEFINING CUSTOM TRAINER WITH WEIGHTED LOSS

✓ Custom Trainer class defined successfully


## Section 3: Define Metrics Computation Function

Implement compute_metrics to calculate accuracy, precision, recall, and F1-score.

In [4]:
print("=" * 80)
print("DEFINING METRICS COMPUTATION FUNCTION")
print("=" * 80)

def compute_metrics(eval_pred):
    """
    Compute evaluation metrics for model predictions.
    
    Args:
        eval_pred: EvalPrediction object containing predictions and labels
    
    Returns:
        Dictionary with accuracy, precision, recall, and F1-score
    """
    predictions, labels = eval_pred
    
    # Get predicted class (argmax of logits)
    pred_labels = np.argmax(predictions, axis=1)
    
    # Compute metrics
    accuracy = accuracy_score(labels, pred_labels)
    precision_macro = precision_score(labels, pred_labels, average='macro', zero_division=0)
    recall_macro = recall_score(labels, pred_labels, average='macro', zero_division=0)
    f1_macro = f1_score(labels, pred_labels, average='macro', zero_division=0)
    
    # Per-class metrics
    precision_weighted = precision_score(labels, pred_labels, average='weighted', zero_division=0)
    recall_weighted = recall_score(labels, pred_labels, average='weighted', zero_division=0)
    f1_weighted = f1_score(labels, pred_labels, average='weighted', zero_division=0)
    
    return {
        'accuracy': accuracy,
        'precision_macro': precision_macro,
        'recall_macro': recall_macro,
        'f1_macro': f1_macro,
        'precision_weighted': precision_weighted,
        'recall_weighted': recall_weighted,
        'f1_weighted': f1_weighted,
    }

print("\n✓ Metrics computation function defined")
print(f"\nMetrics computed:")
print(f"  - Accuracy")
print(f"  - Precision (macro & weighted)")
print(f"  - Recall (macro & weighted)")
print(f"  - F1-score (macro & weighted)")

DEFINING METRICS COMPUTATION FUNCTION

✓ Metrics computation function defined

Metrics computed:
  - Accuracy
  - Precision (macro & weighted)
  - Recall (macro & weighted)
  - F1-score (macro & weighted)


## Section 4: Configure Training Arguments

Set up TrainingArguments with explicit validation configuration.

In [5]:
print("=" * 80)
print("CONFIGURING TRAINING ARGUMENTS")
print("=" * 80)

# Define output directory for model checkpoints
output_dir = "./bert_model_checkpoints"

# Create TrainingArguments with validation configuration
training_args = TrainingArguments(
    # Output and logging configuration
    output_dir=output_dir,
    logging_dir='./logs',
    logging_strategy='epoch',
    log_level='info',
    
    # Training configuration
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=1,
    
    # Learning rate and optimizer
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=100,
    
    # Validation configuration (CRITICAL FOR PROPER VALIDATION)
    eval_strategy='epoch',            # Evaluate at every epoch
    save_strategy='epoch',            # Save checkpoint at every epoch
    
    # Best model selection
    load_best_model_at_end=True,      # Load best model at end
    metric_for_best_model='f1_macro', # Use F1 macro as metric
    greater_is_better=True,           # Higher F1 is better
    save_total_limit=2,               # Keep only 2 best checkpoints
    
    # Misc
    seed=42,
    fp16=torch.cuda.is_available(),   # Mixed precision if GPU available
    dataloader_pin_memory=True,
    remove_unused_columns=False,      # Keep all columns for our custom dataset
)

print(f"\n✓ Training arguments configured")
print(f"\nTraining Configuration:")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Train batch size: {training_args.per_device_train_batch_size}")
print(f"  Eval batch size: {training_args.per_device_eval_batch_size}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Eval strategy: {training_args.eval_strategy}")
print(f"  Save strategy: {training_args.save_strategy}")
print(f"  Load best model: {training_args.load_best_model_at_end}")
print(f"  Best model metric: {training_args.metric_for_best_model}")
print(f"  Output directory: {output_dir}")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


CONFIGURING TRAINING ARGUMENTS

✓ Training arguments configured

Training Configuration:
  Epochs: 3
  Train batch size: 8
  Eval batch size: 8
  Learning rate: 2e-05
  Eval strategy: IntervalStrategy.EPOCH
  Save strategy: SaveStrategy.EPOCH
  Load best model: True
  Best model metric: f1_macro
  Output directory: ./bert_model_checkpoints


## Section 5: Initialize the Trainer

Create Trainer instance with all components including validation dataset.

In [15]:
print("=" * 80)
print("INITIALIZING TRAINER")
print("=" * 80)

# Define a custom data collator for dict-based datasets
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer)

# Initialize custom trainer with weighted loss
trainer = WeightedLossTrainer(
    class_weights=class_weights_tensor,
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,  # IMPORTANT: Pass eval dataset for validation
    compute_metrics=compute_metrics,  # Metrics computation function
    data_collator=data_collator,  # Use proper data collator for dict-based datasets
)

print(f"\n✓ Trainer initialized successfully")
print(f"\nTrainer Configuration:")
print(f"  Model: {type(trainer.model).__name__}")
print(f"  Training dataset: {len(trainer.train_dataset)} samples")
print(f"  Eval dataset: {len(trainer.eval_dataset)} samples")
print(f"  Compute metrics: {trainer.compute_metrics is not None}")
print(f"  Device: {trainer.model.device}")
print(f"  Mixed precision: {training_args.fp16}")

INITIALIZING TRAINER

✓ WeightedLossTrainer initialized
  Class weights: [1.2044225  0.905874   0.93824446]
  Loss function: CrossEntropyLoss (weighted)

✓ Trainer initialized successfully

Trainer Configuration:
  Model: BertForSequenceClassification
  Training dataset: 10240 samples
  Eval dataset: 1284 samples
  Compute metrics: True
  Device: cpu
  Mixed precision: False


## Section 6: Train the Model

Start the training process with validation at each epoch.

In [16]:
print("=" * 80)
print("TRAINING THE MODEL")
print("=" * 80)

print(f"\nStarting training with validation tracking...")
print(f"  Total epochs: {training_args.num_train_epochs}")
print(f"  Evaluation at: Every epoch")
print(f"  Best model selected by: {training_args.metric_for_best_model}")

# Train the model
train_result = trainer.train()

print(f"\n" + "=" * 80)
print("✓ TRAINING COMPLETED SUCCESSFULLY")
print("=" * 80)

# Display training results
print(f"\nTraining Results:")
print(f"  Final training loss: {train_result.training_loss:.6f}")
if hasattr(train_result, 'best_metric'):
    print(f"  Best {training_args.metric_for_best_model}: {train_result.best_metric:.6f}")
if hasattr(train_result, 'best_epoch'):
    print(f"  Best epoch: {int(train_result.best_epoch)}")
print(f"  Training time: {train_result.training_steps} steps")

print(f"\n✓ Best model automatically loaded from checkpoint")

***** Running training *****


TRAINING THE MODEL

Starting training with validation tracking...
  Total epochs: 3
  Evaluation at: Every epoch
  Best model selected by: f1_macro


  Num examples = 10,240
  Num Epochs = 3
  Num update steps per epoch = 1,280
  Instantaneous batch size per device = 8
  Total train batch size (w. parallel, distributed & accumulation) = 8
  Gradient Accumulation steps = 1
  Total optimization steps = 3,840
  Number of trainable parameters = 109,484,547


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

## Section 7: Evaluate on Validation Set

Explicitly evaluate the model on the validation set after training.

In [ ]:
print("=" * 80)
print("EVALUATING ON VALIDATION SET")
print("=" * 80)

# Evaluate on validation set
print(f"\nEvaluating on validation set ({len(eval_dataset)} samples)...")
eval_results = trainer.evaluate(eval_dataset)

print(f"\n✓ Validation Evaluation Results:")
print(f"\n  Loss: {eval_results.get('eval_loss', 'N/A'):.6f}")
print(f"\nAccuracy Metrics:")
print(f"  Accuracy: {eval_results.get('eval_accuracy', 0):.4f}")

print(f"\nPrecision (Macro): {eval_results.get('eval_precision_macro', 0):.4f}")
print(f"Recall (Macro): {eval_results.get('eval_recall_macro', 0):.4f}")
print(f"F1-score (Macro): {eval_results.get('eval_f1_macro', 0):.4f}")

print(f"\nPrecision (Weighted): {eval_results.get('eval_precision_weighted', 0):.4f}")
print(f"Recall (Weighted): {eval_results.get('eval_recall_weighted', 0):.4f}")
print(f"F1-score (Weighted): {eval_results.get('eval_f1_weighted', 0):.4f}")

# Store validation results
val_accuracy = eval_results.get('eval_accuracy', 0)
val_f1 = eval_results.get('eval_f1_macro', 0)

## Section 8: Evaluate on Test Set

Generate predictions on the test set and compute final evaluation metrics without any data leakage.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

print("=" * 80)
print("EVALUATING ON TEST SET")
print("=" * 80)

# Get predictions on test set
print(f"\nGenerating predictions on test set ({len(test_dataset)} samples)...")
predictions = trainer.predict(test_dataset)

# Extract predictions and true labels
test_preds = np.argmax(predictions.predictions, axis=1)
test_labels = predictions.label_ids

print(f"✓ Predictions generated for {len(test_preds)} test samples")

# Compute test metrics
print(f"\nComputing test metrics...")
test_accuracy = np.mean(test_preds == test_labels)

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

test_precision_macro = precision_score(test_labels, test_preds, average='macro', zero_division=0)
test_recall_macro = recall_score(test_labels, test_preds, average='macro', zero_division=0)
test_f1_macro = f1_score(test_labels, test_preds, average='macro', zero_division=0)

test_precision_weighted = precision_score(test_labels, test_preds, average='weighted', zero_division=0)
test_recall_weighted = recall_score(test_labels, test_preds, average='weighted', zero_division=0)
test_f1_weighted = f1_score(test_labels, test_preds, average='weighted', zero_division=0)

print(f"\n✓ Test Set Evaluation Results:")
print(f"\n  Accuracy: {test_accuracy:.4f}")

print(f"\nPrecision (Macro): {test_precision_macro:.4f}")
print(f"Recall (Macro): {test_recall_macro:.4f}")
print(f"F1-score (Macro): {test_f1_macro:.4f}")

print(f"\nPrecision (Weighted): {test_precision_weighted:.4f}")
print(f"Recall (Weighted): {test_recall_weighted:.4f}")
print(f"F1-score (Weighted): {test_f1_weighted:.4f}")

# Compute confusion matrix
print(f"\n" + "=" * 80)
print("CONFUSION MATRIX (Test Set)")
print("=" * 80)
conf_matrix = confusion_matrix(test_labels, test_preds)
label_names = [id_to_label[i] for i in range(len(id_to_label))]

print(f"\nConfusion Matrix:")
print(f"{'':15} {label_names[0]:>12} {label_names[1]:>12} {label_names[2]:>12}")
for i, label in enumerate(label_names):
    print(f"{label:15} {conf_matrix[i, 0]:>12} {conf_matrix[i, 1]:>12} {conf_matrix[i, 2]:>12}")

# Per-class metrics
print(f"\n" + "=" * 80)
print("DETAILED CLASSIFICATION REPORT (Test Set)")
print("=" * 80)
print(classification_report(test_labels, test_preds, target_names=label_names, digits=4))

# Store test results
test_results = {
    'accuracy': test_accuracy,
    'precision_macro': test_precision_macro,
    'recall_macro': test_recall_macro,
    'f1_macro': test_f1_macro,
    'precision_weighted': test_precision_weighted,
    'recall_weighted': test_recall_weighted,
    'f1_weighted': test_f1_weighted
}

## Section 9: Results Summary and Comparison

Compare performance across all three dataset splits (training, validation, test) and analyze model behavior.

In [ ]:
import pandas as pd

print("=" * 80)
print("COMPREHENSIVE RESULTS SUMMARY")
print("=" * 80)

# Extract training metrics from trainer
training_history = trainer.state.log_history
best_epoch = trainer.state.best_model_checkpoint.split('-')[-1] if trainer.state.best_model_checkpoint else 'Final'

print(f"\nTraining Information:")
print(f"  Number of epochs: {training_args.num_train_epochs}")
print(f"  Best model checkpoint: Epoch {best_epoch}")
print(f"  Best F1-macro on validation: {train_result.best_metric:.6f}")
print(f"  Final training loss: {train_result.training_loss:.6f}")

# Compile comparison table
print(f"\n" + "=" * 80)
print("PERFORMANCE COMPARISON ACROSS SPLITS")
print("=" * 80)

comparison_data = {
    'Metric': ['Accuracy', 'Precision (Macro)', 'Recall (Macro)', 'F1 (Macro)',
               'Precision (Weighted)', 'Recall (Weighted)', 'F1 (Weighted)'],
    'Validation': [
        eval_results.get('eval_accuracy', 0),
        eval_results.get('eval_precision_macro', 0),
        eval_results.get('eval_recall_macro', 0),
        eval_results.get('eval_f1_macro', 0),
        eval_results.get('eval_precision_weighted', 0),
        eval_results.get('eval_recall_weighted', 0),
        eval_results.get('eval_f1_weighted', 0)
    ],
    'Test': [
        test_results['accuracy'],
        test_results['precision_macro'],
        test_results['recall_macro'],
        test_results['f1_macro'],
        test_results['precision_weighted'],
        test_results['recall_weighted'],
        test_results['f1_weighted']
    ]
}

comparison_df = pd.DataFrame(comparison_data)
print("\n" + comparison_df.to_string(index=False))

# Calculate differences
print(f"\n" + "=" * 80)
print("GAP ANALYSIS (Test - Validation)")
print("=" * 80)
print(f"\nAccuracy gap: {test_results['accuracy'] - eval_results.get('eval_accuracy', 0):+.4f}")
print(f"F1 (Macro) gap: {test_results['f1_macro'] - eval_results.get('eval_f1_macro', 0):+.4f}")
print(f"Precision (Weighted) gap: {test_results['precision_weighted'] - eval_results.get('eval_precision_weighted', 0):+.4f}")

# Overfitting analysis
val_f1 = eval_results.get('eval_f1_macro', 0)
test_f1 = test_results['f1_macro']
f1_gap = val_f1 - test_f1

print(f"\n" + "=" * 80)
print("GENERALIZATION ANALYSIS")
print("=" * 80)

if f1_gap < -0.05:
    print(f"\n⚠️  Overfitting Detected: Validation F1 > Test F1 by {abs(f1_gap):.4f}")
    print("   Model may not generalize well to new data")
elif f1_gap > 0.05:
    print(f"\n✓ Good Generalization: Test F1 > Validation F1 by {abs(f1_gap):.4f}")
    print("   Model generalizes well to unseen data")
else:
    print(f"\n✓ Balanced Performance: F1 gap of {abs(f1_gap):.4f} indicates good generalization")

# Model information
print(f"\n" + "=" * 80)
print("MODEL INFORMATION")
print("=" * 80)
print(f"\nModel: {training_args.output_dir}")
print(f"Number of parameters: {model.num_parameters():,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"Learning rate: {training_args.learning_rate}")
print(f"Batch size (train): {training_args.per_device_train_batch_size}")
print(f"Batch size (eval): {training_args.per_device_eval_batch_size}")

# Final recommendations
print(f"\n" + "=" * 80)
print("RECOMMENDATIONS")
print("=" * 80)

if test_results['accuracy'] > 0.85:
    print(f"\n✓ Excellent Performance: Test accuracy of {test_results['accuracy']:.4f}")
    print("  Model is ready for production use")
elif test_results['accuracy'] > 0.75:
    print(f"\n✓ Good Performance: Test accuracy of {test_results['accuracy']:.4f}")
    print("  Model can be deployed with monitoring")
else:
    print(f"\n⚠️  Moderate Performance: Test accuracy of {test_results['accuracy']:.4f}")
    print("  Consider improving data or model architecture")

if max(test_results['precision_macro'], test_results['recall_macro']) - min(test_results['precision_macro'], test_results['recall_macro']) > 0.1:
    print(f"\n⚠️  Precision-Recall Imbalance Detected")
    print(f"   Precision: {test_results['precision_macro']:.4f}, Recall: {test_results['recall_macro']:.4f}")
    print("  Consider adjusting decision threshold or class weights")

print(f"\n✓ Training pipeline completed successfully!")
print(f"  Final test F1-score (macro): {test_results['f1_macro']:.4f}")
print(f"  Model saved to: {training_args.output_dir}")